In [17]:
#Imports
import pandas as pd
import datetime
import json
import re
from datasets import load_dataset

In [20]:
#Loading training dataset
train = pd.read_json(
    "/content/train.jsonl",
    lines=True
)

In [21]:
#Sorting the impure data
required_keys = {
    "category",
    "priority",
    "amount_paise",
    "txn_date",
    "txn_id",
    "channel",
    "language",
    "needs_human"
}
allowed_categories = {
    "payment_failed", "refund", "fraud",
    "account_access", "kyc", "offers", "other"
}
allowed_priorities = {"P1", "P2", "P3"}
allowed_channels = {"upi", "card", "netbanking", "wallet"}
allowed_languages = {"en", "hi-en"}
def validate(record):
  error = []
  if not isinstance(record,dict):
    print("Not in JSON format")
  if set(record.keys()) != required_keys:
    error.append("Missing keys")
  if record.get('category') not in allowed_categories:
    error.append("Invalid category")
  if record.get('priority') not in allowed_priorities:
    error.append("Invalid priority")
  amt = record.get('amount_paise')
  if amt is not None and type(amt) is not int:
    error.append("Invalid amount")
  t_date = record.get('txn_date')
  if t_date is not None:
    try:
      new_date = datetime.strptime(t_date, "%Y-%m-%d")
      if new_date > datetime.now():
        error.append("Invalid date")
    except ValueError:
      error.append("Invalid date format")
  if record.get("channel") is not None:
        if record["channel"] not in allowed_channels:
            error.append("Invalid channel")
  if record.get("language") is not None:
        if record["language"] not in allowed_languages:
            error.append("Invalid language")
  if type(record.get("needs_human")) is not bool:
        error.append("needs_human must be boolean")
  txn_id = record.get("txn_id")
  if txn_id is not None:
        if not isinstance(txn_id, str) or not re.fullmatch(
            r"NP\d{10}", txn_id
        ):
            error.append("Invalid transaction ID")
  return error
audit_set = []
for _, rows in train.iterrows():
  try:
    message = rows['messages']
    answer = message[-1]['content']
    label = json.loads(answer)
    error = validate(label)
    if error:
      audit_set.append(
          {
            'id': rows['id'],
            'error': error
          }
      )
  except Exception as e:
    audit_set.append({
        'id': rows['id'],
        'error': [f'Invalid JSON: {str(e)}']
    })

In [22]:
ids = {item['id'] for item in audit_set}
record = train[train['id'].isin(ids)]

In [ ]:
for _, row in record.head(10).iterrows():

    print("ID:", row["id"])

    print("\nTICKET:")
    print(row["messages"][1]["content"])

    print("\nEXISTING LABEL:")
    print(row["messages"][2]["content"])

    print("\n" + "=" * 70)

ID: tr-00001

TICKET:
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.

EXISTING LABEL:
{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}

ID: tr-00003

TICKET:
Received: 2026-09-07 (Mon)
Source: app-chat

Hi, Saamaan wapas kar diya, ab tak paise wapas nahi mile. Android app use kar raha hoon. 5 din pehle ki baat hai. Amount 19151 rupaye tha. Transaction ID NP2416305516 hai. Jaldi solve karo please.

EXISTING LABEL:
{"category": "refund", "priority": "P2", "amount_paise": 1915100, "txn_date": "2026-09-02", "txn_id": "NP2416305516", "channel": null, "language": "hi-en", "needs_human": false}

ID: tr-00004

TICKET:
Received: 

In [ ]:
#cleaned dataset
import csv
cleaning_log = []
cleaned_rows = []

for _, row in train.iterrows():
  row_id = row['id']
  raw_content = row['messages'][2]['content']
  try:
    label = json.loads(raw_content)
  except Exception:
    cleaning_log.append({
        'id': row_id,
        'action': 'drop',
        'reason': 'Truncated or unparseable JSON output',
    })
    continue
  fixed_reasons = []
  if 'prio' in label:
    prio_val = str(label.pop('prio')).lower()
    prio_map = {'low': 'P3', 'medium': 'P2', 'high': 'P1'}
    label['priority'] = prio_map.get(prio_val, 'P3')
    fixed_reasons.append("Renamed 'prio' to 'priority' and mapped value")

  t_date = label.get('txn_date')
  if t_date and re.match(r'^\d{2}/\d{2}/\d{4}$', t_date):
    try:
      parsed_date = datetime.datetime.strptime(t_date, '%d/%m/%Y')
      label['txn_date'] = parsed_date.strftime('%Y-%m-%d')
      fixed_reasons.append('Converted date from DD/MM/YYYY to YYYY-MM-DD')
    except ValueError:
      pass

  txn_id = label.get('txn_id')
  if txn_id and isinstance(txn_id, str):
    norm_id = re.sub(r'[\s\-]', '', txn_id).upper()
    if norm_id != txn_id:
      label['txn_id'] = norm_id
      fixed_reasons.append('Normalized txn_id (removed hyphens/spaces)')

  if label.get('priority') == 'P1' and not label.get('needs_human'):
    label['needs_human'] = True
    fixed_reasons.append('Set needs_human to true because priority is P1')

  if fixed_reasons:
    cleaning_log.append({
        'id': row_id,
        'action': 'fix',
        'reason': '; '.join(fixed_reasons),
    })
  new_messages = list(row['messages'])
  new_messages[2] = {
      'role': 'assistant',
      'content': json.dumps(label, ensure_ascii=False),
  }
  cleaned_rows.append({'id': row_id, 'messages': new_messages})

pd.DataFrame(cleaning_log).to_csv('cleaning_log.csv', index=False)
print(f'Done! Logged {len(cleaning_log)} actions to cleaning_log.csv')

with open('train_cleaned.jsonl', 'w') as f:
  for item in cleaned_rows:
    f.write(json.dumps(item) + '\n')
print(
    f'Saved {len(cleaned_rows)} rows to train_cleaned.jsonl (dropped'
    f' {len(train) - len(cleaned_rows)} rows).'
)

Done! Logged 145 actions to cleaning_log.csv
Saved 1420 rows to train_cleaned.jsonl (dropped 25 rows).


In [4]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 69.5 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [3]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install datasets

In [ ]:
#Using qwen untrained to check the results and score
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from tqdm import tqdm
model_id = "Qwen/Qwen2.5-3B-Instruct"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    trust_remote_code=True,
    device_map="auto",
)
with open("/content/dev.jsonl") as f:
  dev_ticket = [json.loads(line) for line in f]
predictions = []
model.eval()
with torch.no_grad():
  for ticket in tqdm(dev_ticket, desc="Generating predictions"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0][inputs.input_ids.shape[1] :]
    reply_token = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    predictions.append({"id": ticket["id"], "output": reply_token})
with open("/content/my_dev_predictions.jsonl", "w") as f:
  for pred in predictions:
    f.write(json.dumps(pred) + "\n")
print("Saved /content/my_dev_predictions.jsonl!")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Generating predictions: 100%|██████████| 200/200 [37:36<00:00, 11.28s/it]

Saved /content/my_dev_predictions.jsonl!


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/my_dev_predictions.jsonl

group                            n   valid   exact  fields
all                            200   65.5%    0.0%    0.1%
category=account_access         19   73.7%    0.0%    0.0%
category=fraud                  15   66.7%    0.0%    0.0%
category=kyc                    24   91.7%    0.0%    0.0%
category=offers                 18   61.1%    0.0%    0.0%
category=other                  16   87.5%    0.0%    0.0%
category=payment_failed         53   58.5%    0.0%    0.0%
category=refund                 55   52.7%    0.0%    0.5%

per-field accuracy (all rows)
  category         1.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


In [ ]:
#Giving the rules from SCHEMA and checking the output
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from tqdm import tqdm
new_predictions = []
RULES = """You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.

Output:
A single JSON object with exactly these 8 keys. Use null when a value is not present in the ticket:
{"category": "refund", "priority": "P2", "amount_paise": 2936300, "txn_date": "2026-09-13", "txn_id": "NP5972758064", "channel": "netbanking", "language": "en", "needs_human": false}

Allowed values:
- category: payment_failed, refund, fraud, account_access, kyc, offers, other
- priority: P1, P2, P3
- amount_paise: integer in paise (₹1 = 100 paise) or null
- txn_date: YYYY-MM-DD or null
- txn_id: NP followed by 10 digits or null
- channel: upi, card, netbanking, wallet or null
- language: en, hi-en
- needs_human: true, false

Rules:
1. category:
   - fraud: The customer says a transaction happened that they did not make, approve or recognise. This wins over every other category.
   - payment_failed: A payment, transfer or top-up the customer made failed, was declined, is stuck, or did not reach recipient.
   - refund: Money is owed back: asking for a refund, cancelled order, return, charged more than once, refund not arrived.
   - account_access: Cannot get in: locked/blocked account, blocked PIN, forgotten password, OTP not received, reset problems.
   - kyc: Identity verification: how to verify, why needed, failed/pending verification, documents, Aadhaar/PAN, video KYC.
   - offers: Cashback, promo codes, coupons, scratch cards, referral bonuses, reward points.
   - other: Anything else: card delivery, exchange rates, closing account, general how-to questions.

2. priority (apply in this order):
   - P1 if any of these is true: category is fraud, disputed amount is ₹50,000 or more, or customer makes a legal threat (RBI, ombudsman, consumer court, lawyer, legal notice/action).
   - P2 if not P1 and either: category is payment_failed or refund and amount is ₹5,000 or more; or category is account_access.
   - P3 otherwise.

3. Transaction fields:
   - amount_paise, txn_date, txn_id and channel describe one specific transaction. For account_access, kyc and other tickets they are always null.
   - amount_paise: amount of transaction or claim in paise as integer (₹1 = 100 paise). 1.3k = 1,300 INR. 1.5 lakh = 150,000 INR. Ignore balance/limits/fees. null if no amount.
   - txn_date: date as YYYY-MM-DD. Resolve relative dates ('yesterday', '3 days ago', 'kal', 'parso') against Received date. null if not stated.
   - txn_id: NP + 10 digits. Normalize to uppercase with no spaces/hyphens (e.g. NP1234567890). Other bank reference numbers are not txn ids. null if none.
   - channel: payment mode only when explicitly stated: upi, card, netbanking, wallet. null otherwise.

4. language:
   - hi-en if customer message is in Hinglish (Hindi in Latin script).
   - en otherwise.

5. needs_human:
   - true if priority is P1, OR customer says they have already contacted support before ('third time', 'again and again', 'baar baar'). Otherwise false.

Return ONLY the raw JSON object."""
model_id = "Qwen/Qwen2.5-3B-Instruct"
# 1. 4-bit Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
# 2. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    trust_remote_code=True,
    device_map="auto",
)
model.eval()
with open("/content/dev.jsonl") as f:
  dev_ticket = [json.loads(line) for line in f]
with torch.no_grad():
  for ticket in tqdm(dev_ticket, desc="Running Baseline again"):
    user= ticket['messages'][1]['content']
    mess= [
        {'role':'system','content':RULES},
        {'role':'user','content':user}
    ]
    prompts = tokenizer.apply_chat_template(
        mess, tokenize=False, add_generation_prompt=True
    )
    input= tokenizer(prompts, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **input,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_raw_tokens = outputs[0][input.input_ids.shape[1] :]
    raw_token = tokenizer.decode(new_raw_tokens, skip_special_tokens=True).strip()
    new_predictions.append({"id": ticket["id"], "output": raw_token})
  with open("/content/my_secon_dev_predictions.jsonl", "w") as f:
    for pred in new_predictions:
      f.write(json.dumps(pred) + "\n")
  print("Saved /content/my_new_dev_rules_predictions.jsonl!")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Running Baseline again: 100%|██████████| 200/200 [23:45<00:00,  7.13s/it]

Saved /content/my_new_dev_rules_predictions.jsonl!


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/my_secon_dev_predictions.jsonl

group                            n   valid   exact  fields
all                            200  100.0%    7.5%   70.6%
category=account_access         19  100.0%   10.5%   72.4%
category=fraud                  15  100.0%    0.0%   48.3%
category=kyc                    24  100.0%   25.0%   85.4%
category=offers                 18  100.0%    5.6%   79.9%
category=other                  16  100.0%    6.2%   77.3%
category=payment_failed         53  100.0%    3.8%   64.4%
category=refund                 55  100.0%    5.5%   70.5%

per-field accuracy (all rows)
  category        67.5%
  priority        40.0%
  amount_paise    89.0%
  txn_date        65.5%
  txn_id          77.5%
  channel         91.0%
  language        66.0%
  needs_human     68.0%


In [ ]:
#Loading the corrected jsonl
from datasets import load_dataset
train_data = load_dataset(
    "json",data_files="/content/train_cleaned.jsonl",split="train"
)
print(len(train_data))

Generating train split: 0 examples [00:00, ? examples/s]

1420


In [ ]:
!pip install -q --upgrade trl==0.25.1

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
#LoRA constant
LORA_R = 16
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir="./results_qlora",
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.001,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="no",
    push_to_hub=False,
    report_to="none",
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model1 = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=quant_config,
  trust_remote_code=True,
  torch_dtype=torch.float16,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
#Training 1st model
fine_tunning = SFTTrainer(
    model=model1,
    train_dataset=train_data,
    peft_config=lora_parameters,
    args=train_parameters
)

Tokenizing train dataset:   0%|          | 0/1420 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1420 [00:00<?, ? examples/s]

In [ ]:
#Training and saving in local adapter
fine_tunning.train()
fine_tunning.model.save_pretrained("./adapter")
tokenizer.save_pretrained("./adapter")
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM: {peak_vram_gb:.2f} GB")
print("Training finished! Final adapter saved to ./adapter")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.133469
40,1.415998
60,0.748776
80,0.570233
100,0.461547
120,0.448280
140,0.404182
160,0.405769
180,0.385872
200,0.351928


Peak VRAM: 8.76 GB
Training finished! Final adapter saved to ./adapter


In [ ]:
fine_tunning.model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
#Testing 1st model on dev.jsonl and storing in exp1_cleaed_qlora_jsonl
with open("/content/dev.jsonl") as f:
  dev_tickets = [json.loads(line) for line in f]

finetuned_preds = []
fine_tunning.model.eval()

with torch.no_grad():
  for ticket in tqdm(dev_tickets, desc="Evaluating Fine-Tuned Model"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Wrap in autocast to solve the float vs Half dtype mismatch
    with torch.amp.autocast('cuda'):
      outputs = fine_tunning.model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
    new_tokens = outputs[0][inputs.input_ids.shape[1] :]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    finetuned_preds.append({"id": ticket["id"], "output": reply})

with open("/content/exp1_cleaned_qlora.jsonl", "w") as f:
  for pred in finetuned_preds:
    f.write(json.dumps(pred) + "\n")

Evaluating Fine-Tuned Model: 100%|██████████| 200/200 [32:32<00:00,  9.76s/it]


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/exp1_cleaned_qlora.jsonl

group                            n   valid   exact  fields
all                            200  100.0%   93.5%   98.9%
category=account_access         19  100.0%   94.7%   99.3%
category=fraud                  15  100.0%   86.7%   97.5%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   94.3%   98.8%
category=refund                 55  100.0%   87.3%   98.0%

per-field accuracy (all rows)
  category        99.0%
  priority        96.0%
  amount_paise   100.0%
  txn_date        98.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     97.5%


In [13]:
#Saving the model in drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
import torch
from peft import PeftModel
# 1. Mount Google Drive
drive.mount("/content/drive")
adapter_path = "/content/adapter"

# 2. Load the base model in 4-bit
model_id = "Qwen/Qwen2.5-3B-Instruct"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 3. Attach your saved fine-tuned LoRA adapter!
best_model = PeftModel.from_pretrained(base_model, adapter_path)
best_model.eval()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [ ]:
#Creating prediction_test.jsonl from best model
import shutil
from tqdm import tqdm
with open("/content/test_inputs.jsonl") as f:
  test_tickets = [json.loads(line) for line in f]
test_preds = []
best_model.eval()

with torch.no_grad():
  for ticket in tqdm(test_tickets, desc="Generating Test Predictions"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.amp.autocast("cuda"):
      outputs = best_model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
    new_tokens = outputs[0][inputs.input_ids.shape[1] :]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    test_preds.append({"id": ticket["id"], "output": reply})
with open("/content/predictions_test.jsonl", "w") as f:
  for pred in test_preds:
    f.write(json.dumps(pred) + "\n")
shutil.copy(
    "/content/predictions_test.jsonl",
    "/content/drive/MyDrive/NimbusPay_FT_Project/predictions_test.jsonl",
)
print("Saved predictions_test.jsonl locally and to Google Drive!")

In [ ]:
#Checks if the predicted jsonl is in proper format
import json
KEYS = {
    "category",
    "priority",
    "amount_paise",
    "txn_date",
    "txn_id",
    "channel",
    "language",
    "needs_human",
}

with open("/content/predictions_test.jsonl") as f:
  rows = [json.loads(line) for line in f]
valid_json = 0
valid_keys = 0
bad_rows = []

for r in rows:
  try:
    obj = json.loads(r["output"].strip())
    valid_json += 1
    if set(obj.keys()) == KEYS:
      valid_keys += 1
    else:
      bad_rows.append((r["id"], "Wrong keys", obj))
  except Exception:
    bad_rows.append((r["id"], "Invalid JSON", r["output"]))

print(f"Total rows: {len(rows)} (should be 400)")
print(f"Valid JSON: {valid_json}/{len(rows)}")
print(f"All 8 keys present: {valid_keys}/{len(rows)}")

print("\nFirst 3 predictions:")
for r in rows[:3]:
  print(r["id"], "->", r["output"])

if bad_rows:
  print("\nAny broken rows:")
  for b in bad_rows[:5]:
    print(b)

Total rows: 400 (should be 400)
Valid JSON: 400/400
All 8 keys present: 400/400

First 3 predictions:
ts-00001 -> {"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-01", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}
ts-00002 -> {"category": "refund", "priority": "P2", "amount_paise": 1302700, "txn_date": "2026-07-28", "txn_id": "NP5221235957", "channel": "card", "language": "hi-en", "needs_human": false}
ts-00003 -> {"category": "payment_failed", "priority": "P1", "amount_paise": 61875, "txn_date": "2026-09-11", "txn_id": null, "channel": null, "language": "en", "needs_human": true}


In [ ]:
#Training qwen on raw dataset and evaluating on dev.jsonl
import time
raw_train_data = load_dataset(
    "json", data_files="/content/train.jsonl", split="train"
)
print(f"Loaded {len(raw_train_data)} RAW training rows.")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

raw_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
lora_parameters_1 = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)
train_parameters_1 = SFTConfig(
    output_dir="./results_exp2_raw",
    max_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.001,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="no",
    push_to_hub=False,
    report_to="none",
)
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

trainer_raw = SFTTrainer(
    model=raw_model,
    train_dataset=raw_train_data,
    peft_config=lora_parameters_1,
    args=train_parameters_1,
)
trainer_raw.train()
train_minutes = (time.time() - start_time) / 60
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(
    f"\nExp 2 Finished! Time: {train_minutes:.1f} min | Peak VRAM:"
    f" {peak_vram_gb:.2f} GB"
)
with open("/content/dev.jsonl") as f:
  dev_tickets = [json.loads(line) for line in f]

raw_preds = []
trainer_raw.model.eval()
with torch.no_grad():
  for ticket in tqdm(dev_tickets, desc="Evaluating Exp 2 (Raw Data)"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.amp.autocast("cuda"):
      outputs = trainer_raw.model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
    new_tokens = outputs[0][inputs.input_ids.shape[1] :]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    raw_preds.append({"id": ticket["id"], "output": reply})

with open("/content/exp2_raw_qlora.jsonl", "w") as f:
  for pred in raw_preds:
    f.write(json.dumps(pred) + "\n")

Loaded 1445 RAW training rows.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1445 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1445 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.106097
40,1.426063
60,0.773170
80,0.566865
100,0.468895
120,0.430194
140,0.415570
160,0.412229
180,0.403400
200,0.351915



Exp 2 Finished! Time: 41.8 min | Peak VRAM: 8.88 GB


Evaluating Exp 2 (Raw Data): 100%|██████████| 200/200 [32:19<00:00,  9.70s/it]


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/exp2_raw_qlora.jsonl

group                            n   valid   exact  fields
all                            200  100.0%   93.0%   98.8%
category=account_access         19  100.0%   89.5%   98.7%
category=fraud                  15  100.0%   93.3%   99.2%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   92.5%   98.6%
category=refund                 55  100.0%   87.3%   97.7%

per-field accuracy (all rows)
  category       100.0%
  priority        95.0%
  amount_paise   100.0%
  txn_date        98.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     97.0%


In [ ]:
#training qwen for 16bit
Model_16bit = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=quant_config,
  trust_remote_code=True,
  torch_dtype=torch.float16,
)
fine_tunning_16bit = SFTTrainer(
    model=Model_16bit,
    train_dataset=train_data,
    peft_config=lora_parameters,
    args=train_parameters
)
fine_tunning_16bit.train()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.126738


Step,Training Loss
20,3.126738
40,1.409865
60,0.745981
80,0.567222
100,0.460266
120,0.447191
140,0.404645
160,0.405853
180,0.385018
200,0.352142


TrainOutput(global_step=267, training_loss=0.7086717073390546, metrics={'train_runtime': 2131.3742, 'train_samples_per_second': 1.999, 'train_steps_per_second': 0.125, 'total_flos': 1.6841841189126144e+16, 'train_loss': 0.7086717073390546, 'entropy': 0.3383816442003957, 'num_tokens': 754572.0, 'mean_token_accuracy': 0.8844442477932682, 'epoch': 3.0})

In [ ]:
#Checking performance of 16bit fine tunned model using dev.jsonl
with open("/content/dev.jsonl") as f:
  dev_ticket = [json.loads(line) for line in f]

predictions_16bit = []
fine_tunning_16bit.model.eval()

with torch.no_grad():
  for ticket in tqdm(dev_ticket, desc="Evaluating Exp 3 (16-bit LoRA)"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.amp.autocast("cuda"):
      output = fine_tunning_16bit.model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
    new_tokens = output[0][inputs.input_ids.shape[1] :]
    reply_token = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    predictions_16bit.append({"id": ticket["id"], "output": reply_token})

# Save to a dedicated file for Exp 3
with open("/content/exp3_16bit_lora.jsonl", "w") as f:
  for pred in predictions_16bit:
    f.write(json.dumps(pred) + "\n")

print("Saved /content/exp3_16bit_lora.jsonl!")

Evaluating Exp 3 (16-bit LoRA): 100%|██████████| 200/200 [33:30<00:00, 10.05s/it]

Saved /content/exp3_16bit_lora.jsonl!


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/exp3_16bit_lora.jsonl

group                            n   valid   exact  fields
all                            200  100.0%   94.0%   98.9%
category=account_access         19  100.0%   94.7%   99.3%
category=fraud                  15  100.0%   93.3%   99.2%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   96.2%   98.8%
category=refund                 55  100.0%   85.5%   97.5%

per-field accuracy (all rows)
  category        99.0%
  priority        95.0%
  amount_paise   100.0%
  txn_date        99.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     97.5%


In [ ]:
#For my idea
from datetime import datetime
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_NAME = "NIMBUS"

In [ ]:
#For my idea
import wandb
from google.colab import userdata
import os
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mohammed-baqar (mohammed-baqar-cmr-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
#Training model with 2 epoch instead of 3
#also adding wandb to track the training curve in graphical format.
#Can also push on hub but as asked not to do that not doing here
#Can also set quantum config to 8bit
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from tqdm import tqdm
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model2 = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=quant_config,
  trust_remote_code=True,
  torch_dtype=torch.float16,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
#constant
LORA_R = 16
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir="./results_qlora",
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=2,#Changed Hyperparameter
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.001,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="no",
    push_to_hub=False,
    report_to="wandb",#added wandb using forge
)
fine_tunning_2 = SFTTrainer(
    model=model2,
    train_dataset=train_data,
    peft_config=lora_parameters,
    args=train_parameters
)
fine_tunning_2.train()
fine_tunning_2.model.save_pretrained("./adapter_custom")
tokenizer.save_pretrained("./adapter_custom")
print("Training finished! Final adapter saved to ./adapter")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/1420 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1420 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.131873
40,1.410712
60,0.745185
80,0.565852
100,0.461071
120,0.446089
140,0.400981
160,0.400098


Training finished! Final adapter saved to ./adapter


In [ ]:
wandb.finish()

train/entropy,█▆▃▂▁▁▁▁▁
train/epoch,▁▂▃▄▅▅▆▇█
train/global_step,▁▂▃▄▅▅▆▇█
train/grad_norm,█▄▂▃▂▂▁▁
train/learning_rate,▁▃▅▆█▇▄▁
train/loss,█▄▂▁▁▁▁▁
train/mean_token_accuracy,▁▅▇▇█████
train/num_tokens,▁▂▃▄▅▅▆▇█
total_flos,1.1205860345315328e+16
train/entropy,0.38889
train/epoch,2


In [ ]:
#Checking score of custom built model with dev.jsonl
with open("/content/dev.jsonl") as f:
  dev_ticket = [json.loads(line) for line in f]

predictions_custom = []
fine_tunning_2.model.eval()

with torch.no_grad():
  for ticket in tqdm(dev_ticket, desc="Evaluating Exp 4"):
    input_text = ticket["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        input_text, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.amp.autocast("cuda"):
      output = fine_tunning_2.model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
    new_tokens = output[0][inputs.input_ids.shape[1] :]
    reply_token = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    predictions_custom.append({"id": ticket["id"], "output": reply_token})

# Save to a dedicated file for Exp 3
with open("/content/exp4_lora.jsonl", "w") as f:
  for pred in predictions_custom:
    f.write(json.dumps(pred) + "\n")

print("Saved /content/exp4_lora.jsonl!")

Evaluating Exp 4: 100%|██████████| 200/200 [40:28<00:00, 12.14s/it]

Saved /content/exp4_lora.jsonl!


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/exp4_lora.jsonl

group                            n   valid   exact  fields
all                            200  100.0%   84.0%   97.3%
category=account_access         19  100.0%   78.9%   97.4%
category=fraud                  15  100.0%   86.7%   97.5%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   81.1%   96.7%
category=refund                 55  100.0%   70.9%   95.0%

per-field accuracy (all rows)
  category        98.5%
  priority        86.0%
  amount_paise   100.0%
  txn_date        99.0%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     95.0%


In [ ]:
#Error Analysis
import json
import pandas as pd

dev = pd.read_json("/content/dev.jsonl", lines=True)
preds = pd.read_json("/content/exp1_cleaned_qlora.jsonl", lines=True)

for i in range(len(dev)):
  actual = json.loads(dev["messages"][i][2]["content"])
  predicted = json.loads(preds["output"][i])

  if predicted != actual:
    print("ID:", dev["id"][i])
    print("Ticket:", dev["messages"][i][1]["content"])
    print("Predicted:", predicted)
    print("Actual:", actual)
    print("-" * 50)

ID: dv-00006
Ticket: Received: 2026-07-02 (Thu)
Source: twitter

@NimbusPayCare The App shows a fraudulent payment made on my account. It was day before yesterday. You can check np 75453 79360. Amount: 1.3 lakh
Predicted: {'category': 'fraud', 'priority': 'P1', 'amount_paise': 13000000, 'txn_date': '2026-07-01', 'txn_id': 'NP7545379360', 'channel': None, 'language': 'en', 'needs_human': True}
Actual: {'category': 'fraud', 'priority': 'P1', 'amount_paise': 13000000, 'txn_date': '2026-06-30', 'txn_id': 'NP7545379360', 'channel': None, 'language': 'en', 'needs_human': True}
--------------------------------------------------
ID: dv-00011
Ticket: Received: 2026-06-17 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
Why am I missing my refund. App version is 4.2.7. ₹84,137 was the value. This happened yesterday. ID: np3333133441

Regards,
Pooja Nair
Predicted: {'category': 'refund', 'priority': 'P2', 'amount_paise': 8413700, 'txn_date': '2026-06-16', 'txn_id': 'NP3333133441', '